# 03 — Exploratory analysis

Explore labour-market trends by geography, age group, gender, and indicator; check
missingness; and identify the variables that are usable for the dashboard.

> **Governance reminder.** This dashboard produces a *triage* signal for human policy review. It does **not** determine social-assistance need, eligibility, or benefits. Claude outputs are claims, not facts. Missing or suppressed values are flagged, never invented. Any policy interpretation is reviewed by the AI Council before it is treated as decision-ready.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))  # import project modules
import utils
print("Project root:", utils.ROOT)
print("Raw data:    ", utils.RAW)
print("Processed:   ", utils.PROCESSED)
print("Outputs:     ", utils.OUTPUTS)
utils.ensure_dirs()

In [ ]:
import pandas as pd, numpy as np
clean = pd.read_csv(utils.PROCESSED / "labour_force_clean.csv", low_memory=False)
print(f"{len(clean):,} rows · {clean['ref_date'].min()} → {clean['ref_date'].max()}")
print("indicators:", sorted(clean['labour_force_characteristics'].dropna().unique()))
print("genders:", sorted(clean['gender'].dropna().unique()))
print("geographies:", sorted(clean['geo'].dropna().unique()))

## Missingness check (never silently filled)

In [ ]:
miss = clean.groupby("geo")["missing_value_flag"].mean().mul(100).round(2).sort_values(ascending=False)
print("Share of suppressed/missing VALUE cells by geography (%):")
print(miss.to_string())

## Headline unemployment rate by province, latest month

In [ ]:
head = clean[(clean.gender=="Total - Gender") &
             (clean.age_group=="15 years and over") &
             (clean.data_type=="Seasonally adjusted") &
             (clean.labour_force_characteristics=="Unemployment rate")]
latest = head.sort_values("ref_date").groupby("geo").tail(1)
print(latest[["geo","ref_date","value"]].sort_values("value", ascending=False).to_string(index=False))

## Youth vs overall unemployment (latest month)

In [ ]:
def latest_rate(age):
    s = clean[(clean.gender=="Total - Gender") & (clean.age_group==age) &
              (clean.data_type=="Seasonally adjusted") &
              (clean.labour_force_characteristics=="Unemployment rate")]
    return s.sort_values("ref_date").groupby("geo").tail(1).set_index("geo")["value"]
comp = pd.DataFrame({"overall_15plus": latest_rate("15 years and over"),
                     "youth_15_24":  latest_rate("15 to 24 years")})
comp["youth_gap"] = (comp["youth_15_24"] - comp["overall_15plus"]).round(1)
print(comp.sort_values("youth_gap", ascending=False).to_string())

## Optional: figures

`pip install matplotlib` then plot trends. Saved figures live in `outputs/figures/` (see `triage_score_by_province_latest.png`, `unemployment_rate_trend.png`).

In [ ]:
# import matplotlib.pyplot as plt
# for g in ["Canada","Alberta","Newfoundland and Labrador"]:
#     s = head[head.geo==g].sort_values("ref_date")
#     plt.plot(pd.to_datetime(s["ref_date"]), s["value"], label=g)
# plt.legend(); plt.ylabel("Unemployment rate (%)"); plt.show()

### Usable variables for the dashboard

From the LFS alone: unemployment / employment / participation rates (15+ and 15-24), by province, gender, monthly, 1976→present, with trailing-12-month changes. Income, wages, demographics, and affordability add **annual context** once downloaded.